In [ ]:
import bluelib
import time
from pyb import Pin, ADC
import gc

# ==========================================
# 1. HARDWARE CONFIGURATION
# ==========================================
adc = ADC(Pin("X19"))          # LDR analog pin
brgb = bluelib.BRGB(Pin("X1"), 2) # RGB LED module

# ==========================================
# 2. SETTINGS
# ==========================================
SAMPLES = 20
SETTLE_TIME = 0.05
LED_BRIGHTNESS = 255
SAMPLES_PER_COLOR = 10
CSV_FILENAME = 'color_dataset_4f.csv'

# Target classes
COLORS = ['blue', 'red', 'yellow', 'black', 'white', 'green']

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def read_avg():
    total = 0
    for _ in range(SAMPLES):
        total += adc.read()
        time.sleep_ms(2)
    return total / SAMPLES

def turn_off_all():
    brgb.fill((0, 0, 0))
    brgb.write()

def measure_light(r, g, b):
    turn_off_all()
    # GRB format for this specific hardware
    color = (g, r, b)
    brgb.fill(color)
    brgb.write()
    time.sleep(SETTLE_TIME)
    value = read_avg()
    turn_off_all()
    return value

def get_full_profile():
    # 1. Measure background ambient noise
    ambient = measure_light(0, 0, 0)
    
    # 2. Measure RGB reflections sequentially
    r_val = measure_light(LED_BRIGHTNESS, 0, 0) - ambient
    g_val = measure_light(0, LED_BRIGHTNESS, 0) - ambient
    b_val = measure_light(0, 0, LED_BRIGHTNESS) - ambient
    
    # 3. Measure total white light reflection (Distance/Intensity indicator)
    w_val = measure_light(LED_BRIGHTNESS, LED_BRIGHTNESS, LED_BRIGHTNESS) - ambient
    
    # Return sanitized values to prevent negative sensor noise
    return max(r_val, 0), max(g_val, 0), max(b_val, 0), max(w_val, 0)

# ==========================================
# 4. CALIBRATION PHASE
# ==========================================
def calibrate_reference():
    print("=========================================")
    print("   [ REFERENCE CALIBRATION ]             ")
    print("=========================================")
    print("Please place a WHITE surface under the sensor...")
    time.sleep(3)

    ref_r, ref_g, ref_b, ref_w = get_full_profile()
    
    # Prevent division by zero during normalization
    ref_r = max(ref_r, 1.0)
    ref_g = max(ref_g, 1.0)
    ref_b = max(ref_b, 1.0)
    ref_w = max(ref_w, 1.0)

    print("-> Calibration Done!")
    print("   R:{:.1f} | G:{:.1f} | B:{:.1f} | W:{:.1f}".format(ref_r, ref_g, ref_b, ref_w))
    print("=========================================\n")
    return ref_r, ref_g, ref_b, ref_w

# ==========================================
# 5. MAIN DATA COLLECTION
# ==========================================
def main():
    gc.collect()
    
    # Run calibration
    ref_r, ref_g, ref_b, ref_w = calibrate_reference()
    
    # Initialize CSV with 4 features + label
    print("Initializing CSV file...")
    with open(CSV_FILENAME, 'w') as f:
        f.write("Red_Value,Green_Value,Blue_Value,White_Value,label\n")
    
    for color in COLORS:
        print("-----------------------------------------")
        print(">>> Ready to sample: [{}] <<<".format(color.upper()))
        input("Press ENTER when the surface is placed...")
        time.sleep(1) 
        
        # Open file once per color to save flash memory lifespan
        with open(CSV_FILENAME, 'a') as f:
            for sample_idx in range(1, SAMPLES_PER_COLOR + 1):
                raw_r, raw_g, raw_b, raw_w = get_full_profile()
                
                # Normalize features against white reference
                norm_r = raw_r / ref_r
                norm_g = raw_g / ref_g
                norm_b = raw_b / ref_b
                norm_w = raw_w / ref_w
                
                # Write to SD/Flash and flush immediately
                f.write("{:.5f},{:.5f},{:.5f},{:.5f},{}\n".format(
                    norm_r, norm_g, norm_b, norm_w, color
                ))
                f.flush()
                    
                print("   Sample {}/10: R={:.2f} G={:.2f} B={:.2f} W={:.2f}".format(
                    sample_idx, norm_r, norm_g, norm_b, norm_w
                ))
                time.sleep(0.1)
            
        print(">>> Finished logging [{}] <<<\n".format(color.upper()))
        
    print("🎉 DATASET COLLECTION COMPLETE! File: {}".format(CSV_FILENAME))

if __name__ == '__main__':
    main()